# DROID — MACD + Fisher-9 Exhaustion Research v4
## Phase 0: Minimum Viable Research (MVR)

**Date:** 2026-09-29  
**Specification Reference:** v4 §55, §10R, §36R, §57  
**Instrument:** NIFTY 50 (Spot Index)  
**Timeframe:** 5-minute  

---
### Purpose
Produce a definitive preliminary answer to:
> *"Does dual MACD + Fisher-9 extremeness followed by exhaustion contain measurable information about subsequent price movement?"*

…using a clean notebook before investing in database, UI, or production pipeline infrastructure.

In [1]:
import sys
from pathlib import Path

# Ensure repo root in sys.path
REPO_ROOT = Path(".").resolve()
if not (REPO_ROOT / "research").exists():
    REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np

from research.core.fisher import calculate_fisher_point
from research.core.macd import calculate_macd
from research.core.atr import calculate_atr_series
from research.core.validation_tests import (
    verify_lookahead_perturbation,
    run_random_null_test,
    run_planted_signal_test,
    enrich_indicators,
)
from research.core.phase0_runner import run_phase0_ablation

print("Research environment initialized successfully.")

### 1. Data Ingestion & Integrity Check
Loading verified 1-year real historical NIFTY 5m parquet dataset.

In [2]:
data_path = REPO_ROOT / "backend/data/historical/parquet/nifty/5m/candles_v2.parquet"
df = pd.read_parquet(data_path)
print(f"Loaded {len(df):,} bars from {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

### 2. Validation Proofs (§55 Deliverables 0.3, 0.4, 0.5)
Testing Look-Ahead Perturbation, Random-Null GBM, and Planted-Signal Recovery.

In [3]:
# Deliverable 0.3: Look-Ahead Perturbation Test
lookahead_pass = verify_lookahead_perturbation(df, test_indices=[50, 100, 250, 500, 1000])
print(f"Deliverable 0.3 (Look-Ahead Proof): Passed = {lookahead_pass}")

# Deliverable 0.4: Random-Null Test
null_res = run_random_null_test(n_bars=5000, seed=42)
print(f"Deliverable 0.4 (Random-Null Test): Spurious Edge = {null_res['significant']} (p-val: {null_res['p_value']:.4f})")

# Deliverable 0.5: Planted-Signal Test
planted_res = run_planted_signal_test(n_bars=5000, planted_effect_atr=1.2, seed=123)
print(f"Deliverable 0.5 (Planted Signal Test): Recovered = {planted_res['recovered']} (p-val: {planted_res['p_value']:.4e})")

### 3. Stage Ablation A through F on Real Data (§36R, §57)
Computing forward outcomes, effective N clustering, power evaluation, and Benjamini-Hochberg FDR q-values.

In [4]:
ablation_df, summary = run_phase0_ablation(df, horizon_bars=6)
display_cols = [
    "stage_label", "n_raw", "n_eff", "power_category", "observed_power",
    "baseline", "lift", "ci_lower", "ci_upper", "q_value", "direction"
]
ablation_df[display_cols]

### 4. Gate 0 Decision & Conclusion
Evaluating empirical evidence against §55 Go/No-Go criteria.

In [5]:
print("=" * 70)
print(f"PHASE 0 GATE 0 DECISION: >>> {summary['decision']} <<<")
print(f"Rationale: {summary['rationale']}")
print("=" * 70)